# 4. Transcribe and write the release

1. **Transcripts.** The audio of every segment (`work/audio/`, on the video's clock) is
   transcribed with WhisperX: speech found by Silero VAD (pauses of 1 s or more split it),
   each piece's language detected by Whisper large-v3-turbo and the non-Romanian ones
   silenced, then Whisper large-v3 in Romanian, then word timestamps by WhisperX's Romanian
   wav2vec2 alignment, which also splits the text into sentences.
2. **Filters** (`common.keep_sentences`): sentences under 2 s, with under 0.1 s or over 2 s per
   word, Whisper's repetition loops and its sign-off ("mulțumim pentru vizionare") are
   dropped. Then sentences with letters outside Romanian go to a language model with the
   Whisper text of their chunk and an independent transcription of it (NeMo, SpeD ParakeetRo):
   the chunk's sentences are dropped if the model calls it a foreign-language passage. The
   YouTube sign-offs Whisper also makes up ("nu uitați să dați like…", `common.YOUTUBE`), and
   the sentences with letters of another script than Latin (Whisper half-transcribing Russian or
   Ukrainian speech, `common.other_script`), are left out of the labels.
3. **Labels**: the sentences of each released segment, timed from its first frame, dropping
   those not on its video; segments left without a sentence are dropped. A segment that is part
   of a range (split or cut short) keeps the sentences of its part.
4. **Tokenizer**: a SentencePiece unigram model of 16,384 pieces over the lowercased sentences
   of the labels.
5. **Splits**: per channel, whole broadcasts drawn at random (seed 0) until test and val each
   hold 5% of the labelled hours. A broadcast keeps the split it was drawn into.

Output: `work/release/` with `manifests/`, `dataset/` (labels) and `tokenizer/`, the layout of
the repository and of the Hugging Face dataset.

1 and 2 are done for this run (`work/`); the release's transcripts, model verdicts and splits
are in `frozen/`, and an update adds to them without changing the splits drawn. 'replay'
compares the result with the repository. The audio of the first ranges transcribed was cut
together with the video by stream copy, so it started at the keyframe before the first frame and
its gaps were closed up; their transcripts carry, as `clock`, where that audio started and where
its gaps moved it, measured against the video, so that they land on the right frames.

In [ ]:
import json
import os
import random
import shutil
from pathlib import Path

from common import CHANNELS, FROZEN, MODE, REPO, WORK, YOUTUBE, cut, found, keep_sentences, label_path, other_script, \
    place, read, write

# The original run's settings for its 12 GB GPU. With less memory, lower the batches (a CUDA error
# can leave the GPU unusable until the kernel restarts) or use 'int8_float16', which differs slightly.
COMPUTE_TYPE = 'float16'
LANGUAGE_BATCH, TRANSCRIBE_BATCH = 96, 32
RELEASE = WORK / 'release'
items = read(RELEASE / 'manifests.json')

## 1. Transcripts

Each file's language mask, Whisper chunks and aligned sentences are saved next to it in
`work/asr/` as they are made; a stopped run resumes from there. Then they are gathered in
`work/asr.json.zst` (per segment, the aligned sentences [start, end, text, words]) and
`work/chunks.json.zst` (Whisper's chunks [start, end, text], for step 2). A batch that runs out
of GPU memory is halved and run again.

In [ ]:
if MODE != 'replay':
    import gc

    import numpy as np
    import torch
    import whisperx
    from whisperx.audio import N_SAMPLES, log_mel_spectrogram

    AUDIO, OUT = WORK / 'audio', WORK / 'asr'
    files = sorted(AUDIO.glob('*/**/*.flac'))
    done = lambda path, kind: (OUT / path.relative_to(AUDIO)).with_suffix(kind)

    def retry(step, batch):
        """step(batch), halving the batch while the GPU runs out of memory."""
        while True:
            try:
                return step(batch)
            except RuntimeError as e:
                if 'out of memory' not in str(e).lower() and 'cuda' not in str(e).lower() or batch <= 1:
                    raise
                batch //= 2
                gc.collect(); torch.cuda.empty_cache()

    def masked(path):
        wav = whisperx.load_audio(str(path))
        mask = np.unpackbits(np.load(done(path, '.mask.npy')))[:len(wav)].astype(bool)
        wav[~mask] = 0.0
        return wav

    # language mask: Silero VAD pieces, Whisper large-v3-turbo's language of each
    todo = [p for p in files if not done(p, '.mask.npy').exists()]
    if todo:
        turbo = whisperx.load_model('large-v3-turbo', 'cuda', compute_type='float16')
        vad, utils = torch.hub.load('snakers4/silero-vad', 'silero_vad', trust_repo=True)
        vad, speech_timestamps = vad.to('cuda'), utils[0]
        n_mels = turbo.model.feat_kwargs.get('feature_size') or 80
        for path in todo:
            wav = whisperx.load_audio(str(path))
            pieces = [(s['start'], s['end']) for s in speech_timestamps(
                torch.from_numpy(wav).to('cuda'), vad, sampling_rate=16000, min_silence_duration_ms=1000, speech_pad_ms=0)]

            def romanian(batch):
                mask = np.ones(len(wav), bool)
                for i in range(0, len(pieces), batch):
                    mels = [log_mel_spectrogram(wav[a:b][:N_SAMPLES], n_mels=n_mels, padding=max(0, N_SAMPLES - (b - a)))
                            for a, b in pieces[i:i + batch]]
                    with torch.no_grad():
                        found = turbo.model.model.detect_language(turbo.model.encode(torch.stack(mels)))
                    for (a, b), result in zip(pieces[i:i + batch], found):
                        if result[0][0][2:-2] != 'ro':
                            mask[a:b] = False
                return mask
            done(path, '.mask.npy').parent.mkdir(parents=True, exist_ok=True)
            np.save(done(path, '.mask.npy'), np.packbits(retry(romanian, LANGUAGE_BATCH)))
            print('language', path.relative_to(AUDIO), flush=True)
        del turbo, vad
        gc.collect(); torch.cuda.empty_cache()

    # Whisper large-v3, in Romanian
    todo = [p for p in files if not done(p, '.whisper.json').exists()]
    if todo:
        big = whisperx.load_model('large-v3', 'cuda', compute_type=COMPUTE_TYPE)
        for path in todo:
            wav = masked(path)
            write(done(path, '.whisper.json'),
                  retry(lambda batch: big.transcribe(wav, batch_size=batch, language='ro')['segments'], TRANSCRIBE_BATCH))
            print('transcribed', path.relative_to(AUDIO), flush=True)
        del big
        gc.collect(); torch.cuda.empty_cache()

    # word timestamps and sentences: WhisperX's Romanian wav2vec2 alignment
    todo = [p for p in files if not done(p, '.aligned.json').exists()]
    if todo:
        aligner, metadata = whisperx.load_align_model(language_code='ro', device='cuda')
        for path in todo:
            wav, chunks = masked(path), read(done(path, '.whisper.json'))
            aligned = whisperx.align(chunks, aligner, metadata, wav, 'cuda', return_char_alignments=False)
            write(done(path, '.aligned.json'), [[s['start'], s['end'], s['text'], len(s['words'])] for s in aligned['segments']])
            print('aligned', path.relative_to(AUDIO), flush=True)
        del aligner
        gc.collect(); torch.cuda.empty_cache()

    asr, chunks = {c: {} for c in CHANNELS}, {c: {} for c in CHANNELS}
    for path in files:
        channel, *rest = path.relative_to(AUDIO).with_suffix('').parts
        asr[channel]['/'.join(rest)] = read(done(path, '.aligned.json'))
        chunks[channel]['/'.join(rest)] = [[c['start'], c['end'], c['text']] for c in read(done(path, '.whisper.json'))]
    write(WORK / 'asr.json.zst', asr)
    write(WORK / 'chunks.json.zst', chunks)

## 2. Foreign-language passages

A sentence goes with the Whisper chunk it overlaps most (by half its length at least), one that
contains its text if there is one. NeMo transcribes those chunks; the language model
(Azure OpenAI, `gpt-5-mini`; set `AZURE_OPENAI_ENDPOINT` and `AZURE_OPENAI_API_KEY`) answers
with JSON. Output: `work/llm.json`, per segment the dropped sentences (their index after the
filters) and the model's reasons.

In [ ]:
ROMANIAN_LETTERS = set('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZăĂâÂîÎșȘțȚşŞţŢ')
PROMPT = """
You are a Romanian ASR quality analyst.

Task:
Given suspicious sentence(s), a full WhisperX chunk, and its parallel NeMo chunk, decide if this is a foreign-language hallucination.

Important:
- Do NOT flag normal Romanian sentences that contain a foreign proper name, brand, loanword, acronym, or single foreign token.
- Only flag when the sentence/chunk is genuinely another language (or gibberish trying to transcribe another language), not Romanian with occasional terms.

Return STRICT JSON with keys:
- sentence_hallucination: boolean
- chunk_hallucination: boolean
- reason: short string
""".strip()


def chunk_of(sentence, chunks):
    """Index of the chunk holding the sentence, or -1."""
    a, b = sentence['start'], sentence['end']
    best, best_overlap = -1, 0
    for require_text in (True, False):
        for i, (c0, c1, text) in enumerate(chunks):
            overlap = max(0, min(b, c1) - max(a, c0))
            if (overlap / max(b - a, 0.001) >= 0.5 and overlap > best_overlap
                    and (not require_text or (sentence['text'].strip() and sentence['text'].strip() in text))):
                best, best_overlap = i, overlap
        if best >= 0:
            return best
    return -1


if MODE != 'replay':
    import time
    import tempfile

    import nemo.collections.asr as nemo_asr
    import soundfile
    import whisperx
    from huggingface_hub import hf_hub_download
    from openai import AzureOpenAI

    asr, chunks = read(WORK / 'asr.json.zst'), read(WORK / 'chunks.json.zst')
    nemo = nemo_asr.models.ASRModel.restore_from(hf_hub_download(
        'gabrielpirlo/SpeD_ParakeetRo_110M_TDT-CTC', 'SpeD-ParakeetRo_110M_TDT-CTC.nemo')).to('cuda').eval()
    client = AzureOpenAI(azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'], api_key=os.environ['AZURE_OPENAI_API_KEY'],
                         api_version='2024-12-01-preview')

    def verdict(suspects, whisper_text, nemo_text):
        """(foreign, reason); a failed request is retried, and the run stops if the model does not answer."""
        for attempt in range(6):
            try:
                answer = client.chat.completions.create(
                    model='gpt-5-mini', response_format={'type': 'json_object'},
                    messages=[{'role': 'system', 'content': PROMPT},
                              {'role': 'user', 'content': f'Suspicious sentence(s):\n{json.dumps(suspects, ensure_ascii=False, indent=2)}'
                                                          f'\n\nWhisperX chunk:\n{whisper_text}\n\nNeMo chunk:\n{nemo_text}'}])
                v = json.loads(answer.choices[0].message.content)
                return bool(v.get('sentence_hallucination')) or bool(v.get('chunk_hallucination')), str(v.get('reason', ''))
            except Exception as e:
                error = e
                time.sleep(2 ** attempt)
        raise RuntimeError(f'the language model did not answer: {error}')

    llm, asked = {c: {} for c in CHANNELS}, 0
    for channel in CHANNELS:
        for key, rows in asr[channel].items():
            sentences = keep_sentences(rows)
            where = [chunk_of(s, chunks[channel][key]) for s in sentences]
            suspect = {}
            for i, s in enumerate(sentences):
                if where[i] >= 0 and any(ch.isalpha() and ch not in ROMANIAN_LETTERS for ch in s['text']):
                    suspect.setdefault(where[i], []).append(s['text'])
            if not suspect:
                continue
            wav = whisperx.load_audio(str(WORK / 'audio' / channel / f'{key}.flac'))
            tmp = Path(tempfile.mkdtemp())
            for c in suspect:
                c0, c1, _ = chunks[channel][key][c]
                soundfile.write(tmp / f'{c}.wav', wav[int(c0 * 16000):int(c1 * 16000)], 16000)
            heard = nemo.transcribe([str(tmp / f'{c}.wav') for c in suspect], batch_size=16, verbose=False)
            heard = heard[0] if isinstance(heard, tuple) else heard
            shutil.rmtree(tmp)
            dropped, reasons = set(), []
            asked += len(suspect)
            for c, h in zip(suspect, heard):
                bad, reason = verdict(suspect[c], chunks[channel][key][c][2], getattr(h, 'text', h).strip())
                if bad:
                    dropped.add(c)
                    reasons.append(reason)
            if dropped:
                llm[channel][key] = {'removed': [i for i, c in enumerate(where) if c in dropped], 'reasons': reasons}
    write(WORK / 'llm.json', llm)
    print(asked, 'chunks asked about,', sum(len(v['reasons']) for c in llm.values() for v in c.values()), 'foreign')

## The release's transcripts

The sentences of each transcribed range that pass the filters, on their audio's clock:
`frozen/transcripts.json.zst`, and in an update this run's (`work/asr.json.zst`, `work/llm.json`).
A broadcast done again replaces the transcripts of the ranges whose frames it changed.

In [ ]:
release = found()
texts, recorded = ({c: {} for c in CHANNELS} for _ in range(2)) if MODE == 'live' else \
    (read(FROZEN / 'transcripts.json.zst'), read(FROZEN / 'splits.json'))
if MODE != 'replay':
    asr, llm = ((read(WORK / name) if (WORK / name).exists() else {}) for name in ('asr.json.zst', 'llm.json'))
    before = read(FROZEN / 'segments.json') if MODE == 'update' else {c: {} for c in CHANNELS}
    for c, made in read(WORK / 'segments.json').items():
        for key, now in made.items():             # done again: drop the transcripts of the ranges that changed
            earlier = before[c].get(key) or {}
            old = (earlier.get('ranges') or []) if earlier.get('interpreter') else []
            same = {i for i, r in enumerate((now or {}).get('ranges') or []) if i < len(old) and old[i] == r}
            for k in [k for k in texts[c] if k.rsplit('/', 1)[0] == key and int(k.rsplit('/', 1)[1]) not in same]:
                del texts[c][k]
        for k, rows in asr.get(c, {}).items():
            texts[c][k] = {'sentences': rows, 'removed': llm.get(c, {}).get(k, {}).get('removed', [])}
transcripts = {c: {k: [s for i, s in enumerate(keep_sentences(t['sentences'])) if i not in set(t['removed'])]
                   for k, t in texts[c].items()} for c in CHANNELS}
clocks = {c: {k: t['clock'] for k, t in texts[c].items() if 'clock' in t} for c in CHANNELS}

## 3. Labels

`common.place` times the sentences from the segment's first frame and drops those that start
before it or at/after its last frame, or end more than 0.1 s past it (ends up to 0.1 s past it
are clipped).

In [ ]:
labels = {}
for channel in CHANNELS:
    for it in items[channel]:
        broadcast = key = it['id'] if channel == 'PrimaTV' else it['date']
        if channel == 'ProTV':      # the broadcast's key in segments.json: <date>/<file or episode>
            key = next(k for k in release['ProTV'] if k.startswith(broadcast + '/') and release['ProTV'][k]
                       and 'unavailable' not in release['ProTV'][k])
        kept, ranges = [], release[channel][key]['ranges']
        shifts = release[channel][key].get('shift') or [0] * len(ranges)
        for seg in it['segments']:   # its range: the one it lies in (3_verify can cut a range, or split it in parts)
            i = next(i for i, ((s, e), d) in enumerate(zip(ranges, shifts)) if s + d <= seg['start'] and seg['end'] <= e + d)
            (start, end), d = ranges[i], shifts[i]
            said = [s for s in transcripts[channel][f'{key}/{i}']
                    if not YOUTUBE.search(s['text']) and not other_script(s['text'])]
            sentences = place(said, end - start, clocks[channel].get(f'{key}/{i}'))
            first, last = seg['start'] - (start + d), (end + d) - seg['end']     # frames cut by 3_verify's decisions
            if first or last:
                sentences = cut(sentences, first, end - start - first - last)
            if sentences:
                labels[label_path(channel, broadcast, seg['index'])] = sentences
                kept.append(seg)
        it['segments'] = kept
    items[channel] = [it for it in items[channel] if it['segments']]

## 4. Tokenizer

Trained on the lowercased sentences of the labels.

In [ ]:
import sentencepiece as spm

tokenizer = RELEASE / 'tokenizer'
tokenizer.mkdir(parents=True, exist_ok=True)
with open(tokenizer / 'sentences_lowercase.txt', 'w', encoding='utf-8') as f:
    for path in sorted(labels):
        f.writelines(s['text'].lower() + '\n' for s in labels[path])
here = os.getcwd()
os.chdir(tokenizer)            # the model records its input's and its own file name
spm.SentencePieceTrainer.Train(input='sentences_lowercase.txt', model_prefix='spm_unigram_lowercase_16k',
                               model_type='unigram', vocab_size=16384, character_coverage=1.0,
                               shuffle_input_sentence=True, unk_id=0, unk_piece='<unk>', pad_id=1, pad_piece='<pad>',
                               bos_id=2, bos_piece='<s>', eos_id=3, eos_piece='</s>', minloglevel=2)
os.chdir(here)
(tokenizer / 'sentences_lowercase.txt').unlink()

## 5. Splits and the release

A broadcast is drawn into a split once, and keeps it (`splits.json` of the run that drew it): a
broadcast done again later, whose hours change, stays where it was. Those without one are drawn
in a seeded random order: into test while the channel's test broadcasts hold less than 5% of its
labelled hours, then val likewise, then train.

In [ ]:
MANIFEST = {'PrimaTV': 'prima', 'ProTV': 'protv', 'Digi24': 'digi24'}


def hours(it, channel):
    return sum(s['end'] - s['start'] for seg in it['segments']
               for s in labels[label_path(channel, it.get('id') or it['date'], seg['index'])]) / 3600


def splits(broadcasts, channel, drawn):
    """Per broadcast: test, val or train. drawn: (hours, split) of the channel's broadcasts already in one."""
    hours_of = [hours(it, channel) for it in broadcasts]
    order = list(range(len(broadcasts)))
    random.Random(0).shuffle(order)
    total, split = sum(hours_of) + sum(h for h, _ in drawn), {}
    held = {k: sum(h for h, s in drawn if s == k) for k in ('test', 'val')}
    for i in order:
        split[i] = 'test' if held['test'] < 0.05 * total else 'val' if held['val'] < 0.05 * total else 'train'
        if split[i] != 'train':
            held[split[i]] += hours_of[i]
    return [split[i] for i in range(len(broadcasts))]


for channel in CHANNELS:
    split = {id(it): recorded[channel].get(it.get('id') or it['date']) for it in items[channel]}
    new = [it for it in items[channel] if not split[id(it)]]
    if new:
        split.update(zip(map(id, new), splits(new, channel, [(hours(it, channel), split[id(it)])
                                                             for it in items[channel] if split[id(it)]])))
    items[channel] = [{**{k: v for k, v in it.items() if k != 'segments'}, 'split': split[id(it)], 'segments': it['segments']}
                      for it in items[channel]]
    write(RELEASE / 'manifests' / f'{MANIFEST[channel]}_manifest.json', {'items': items[channel]}, indent=2)
shutil.rmtree(RELEASE / 'dataset', ignore_errors=True)    # no label of an earlier attempt stays
for path, sentences in labels.items():
    write(RELEASE / 'dataset' / path, sentences, indent=2, newline=False)
print({c: (len(items[c]), sum(len(it['segments']) for it in items[c])) for c in CHANNELS}, len(labels), 'label files')

## Compare with the repository

In [ ]:
if MODE == 'replay':
    same = lambda a, b: Path(a).read_bytes() == Path(b).read_bytes()
    for channel, name in MANIFEST.items():
        print(f'manifests/{name}_manifest.json identical:', same(RELEASE / 'manifests' / f'{name}_manifest.json',
                                                                REPO / 'manifests' / f'{name}_manifest.json'))
    print('tokenizer identical:', same(tokenizer / 'spm_unigram_lowercase_16k.model',
                                       REPO / 'tokenizer' / 'spm_unigram_lowercase_16k.model'))
    repo = {str(p.relative_to(REPO / 'dataset')) for p in (REPO / 'dataset').glob('*/**/segment_*.json')}
    identical = sum(same(RELEASE / 'dataset' / path, REPO / 'dataset' / path) for path in labels)
    print(f'label files: {len(labels)} built, {len(repo)} in the repository, same set: {set(labels) == repo}, '
          f'identical: {identical}')

## Keep the update

In 'update' mode, once the release is checked and the decisions made, set `FREEZE = True`: this
run's outputs join `frozen/`, where 'replay' finds them. Then copy `release/` to the repository
and set the mode back to 'replay'.

In [ ]:
FREEZE = False
if MODE == 'update' and FREEZE:
    write(FROZEN / 'segments.json', release)
    write(FROZEN / 'transcripts.json.zst', texts)
    write(FROZEN / 'splits.json', {c: {**recorded[c], **{it.get('id') or it['date']: it['split'] for it in items[c]}}
                                   for c in CHANNELS})
    lists, listed = read(FROZEN / 'broadcasts.json'), read(WORK / 'broadcasts.json')
    done = {c: {key.split('/')[0] for key in read(WORK / 'segments.json')[c]} for c in ('ProTV', 'Digi24')}
    for route, listing in listed['ProTV'].items():          # the lists of the broadcasts this run did
        lists.setdefault('ProTV', {}).setdefault(route, {}).update({d: v for d, v in listing.items() if d in done['ProTV']})
    days = {d: v for d, v in listed['Digi24']['listing'].items() if d in done['Digi24']}
    lists['Digi24']['listing'].update(days)
    lists['Digi24']['video_src'].update({u: listed['Digi24']['video_src'][u] for us in days.values() for u in us})
    write(FROZEN / 'broadcasts.json', {c: {p: dict(sorted(v.items())) for p, v in parts.items()} for c, parts in lists.items()})
    print('frozen')